# Practical Exercise no. 2

### T1
Load the datasets and merge them by “Area Code” and “Phone”. Then transform the “Churn” attribute
into a nominal attribute;

In [ ]:
import pandas as pd

url_calls = 'datasets/CallsData.csv' 
url_contract = 'datasets/ContractData.csv'

calls_df = pd.read_csv(url_calls)
contract_df = pd.read_csv(url_contract)

df_merged = pd.merge(calls_df, contract_df, on=["Area Code", "Phone"])

df_merged['Churn'] = df_merged['Churn'].astype(str)

### T2
Use a Decision Tree as a classifier (sklearn.tree.DecisionTreeClassifier) and evaluate the model’s accuracy
in predicting the churn. Also evaluate the model using the f1_macro metric. Use 10-fold cross validation ;
**Note:** *define X and y, paying attention to the type of attributes that are part of X.*


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate
import numpy as np

y = df_merged['Churn']

X = df_merged.drop(columns=['Churn', 'Phone'])

X = pd.get_dummies(X)

clf = DecisionTreeClassifier(random_state=42)

scoring = ['accuracy', 'f1_macro']
cv_results = cross_validate(clf, X, y, cv=10, scoring=scoring)

print(f"Accuracy média: {np.mean(cv_results['test_accuracy']):.2f}")
print(f"F1 Macro médio: {np.mean(cv_results['test_f1_macro']):.2f}")

Accuracy média: 0.91
F1 Macro médio: 0.83


### T3 
Obtain the model's confusion matrices and analyze them critically. What conclusions can be drawn?

In [3]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

y_pred = cross_val_predict(clf, X, y, cv=10)

cm = confusion_matrix(y, y_pred)
print("Matriz de Confusão:\n", cm)
print("\nRelatório de Classificação:\n", classification_report(y, y_pred))

Matriz de Confusão:
 [[2697  153]
 [ 139  344]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.95      0.95      0.95      2850
           1       0.69      0.71      0.70       483

    accuracy                           0.91      3333
   macro avg       0.82      0.83      0.83      3333
weighted avg       0.91      0.91      0.91      3333



### T4
Change the hyperparameters of the Decision Tree (*criterion* and *max_depth*). What variation in model
performance is underlying these changes?


In [4]:
criterios = ['gini', 'entropy']
profundidades = [None, 3, 5, 10] # 'None' significa que a árvore cresce até ao limite

for crit in criterios:
    for depth in profundidades:
        clf_tuned = DecisionTreeClassifier(criterion=crit, max_depth=depth, random_state=42)
        
        cv_res = cross_validate(clf_tuned, X, y, cv=10, scoring=['accuracy', 'f1_macro'])
        acc = np.mean(cv_res['test_accuracy'])
        f1 = np.mean(cv_res['test_f1_macro'])
        
        print(f"Critério: {crit:7} | Max Depth: {str(depth):4} -> Acc: {acc:.3f} | F1 Macro: {f1:.3f}")

Critério: gini    | Max Depth: None -> Acc: 0.912 | F1 Macro: 0.825
Critério: gini    | Max Depth: 3    -> Acc: 0.902 | F1 Macro: 0.737
Critério: gini    | Max Depth: 5    -> Acc: 0.935 | F1 Macro: 0.857
Critério: gini    | Max Depth: 10   -> Acc: 0.936 | F1 Macro: 0.863
Critério: entropy | Max Depth: None -> Acc: 0.920 | F1 Macro: 0.840
Critério: entropy | Max Depth: 3    -> Acc: 0.902 | F1 Macro: 0.743
Critério: entropy | Max Depth: 5    -> Acc: 0.935 | F1 Macro: 0.859
Critério: entropy | Max Depth: 10   -> Acc: 0.939 | F1 Macro: 0.869
